# 06 — The disease-trajectory graph (issue #10)

## If you're new to this project, start here

This project studies **patient-similarity graphs**: connecting patients who look medically alike, to see whether that helps predict things like long hospital stays or new hypertension diagnoses. Every notebook before this one (00-05) built and tested pieces of that patient graph.

**This notebook builds a completely different kind of graph.** Instead of connecting *patients* to *patients*, it connects *diagnoses* to *diagnoses* -- an edge from "diabetes" to "kidney disease" means "having diabetes measurably raises the chance of later developing kidney disease." This is called a **disease-trajectory graph**, and its purpose is different too: it's not used to make predictions. It's an *explanatory* graph, meant to visually tell a real story about how conditions progress into each other over time in this patient population, and it's the input for a specific clustering technique (Girvan-Newman, coming in issue #14) that groups diagnoses into families based on how they connect.

## The idea, in plain terms

For every pair of diagnoses A and B, we ask one question: **"among patients who have A, how much more likely are they to later get B, compared to a random patient in the whole population?"** That's a statistic called **relative risk (RR)**:

> RR = 3 means "having A makes you 3x more likely to later get B than a random person off the street."
> RR = 1 means "no relationship -- B follows A exactly as often as you'd expect by pure chance."
> RR < 1 means "having A actually makes B *less* likely" (rare, but possible).

We only keep a connection in the graph if two things are both true:
1. **Enough people actually showed this pattern** (not just 2 patients by coincidence) -- called the *minimum support* requirement.
2. **The pattern is statistically real**, checked with a formal significance test, made *stricter* the more comparisons we run (called a **Bonferroni correction** -- since we're testing thousands of diagnosis pairs at once, some will look "significant" by pure chance alone unless we raise the bar to compensate).

**One honest tradeoff, stated up front, not hidden:** relative risk here is a *simplified*, population-baseline comparison (not a rigorous causal/epidemiological estimate with a matched control group) -- appropriate for an explanatory graph in this project, not a clinical research claim. We say "elevates risk," not "causes."

In [ ]:
import sys
from pathlib import Path

_repo_root = Path.cwd()
while not (_repo_root / "pyproject.toml").exists() and _repo_root != _repo_root.parent:
    _repo_root = _repo_root.parent
sys.path.insert(0, str(_repo_root))

from scripts.local_setup import ensure_local_data  # noqa: E402

data_root = ensure_local_data()
assets = data_root / "EHRSHOT_ASSETS"
meds_reader_path = data_root / "meds_reader_omop_ehrshot"

## What each import below is for

- `community` (the `python-louvain` package) -- finds clusters of densely-interconnected diagnoses, purely for a richer, more readable picture in this notebook (the "official" clustering method for this project, Girvan-Newman, is issue #14's job -- this is a preview, not a replacement).
- `ehrgraph.data.vocabulary` -- the same real-disease-name lookup built in issue #3, reused here so every diagnosis on every chart has an actual name, never a bare code.
- `ehrgraph.graphs.trajectory` -- the module built for this issue: everything from "extract each patient's diagnosis timeline" through "build the final graph" lives there, tested independently in `tests/test_trajectory.py`.

In [ ]:
import community as community_louvain
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd

from ehrgraph.data.cohort import patient_code_counts
from ehrgraph.data.loader import open_database
from ehrgraph.data.vocabulary import load_concept_table, resolve_codes
from ehrgraph.graphs.trajectory import (
    bonferroni_significant,
    build_trajectory_graph,
    compute_relative_risks,
    first_occurrence_times,
    symmetrize_max,
)

plt.rcParams["figure.facecolor"] = "white"
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False
PALETTE = [
    "#4C72B0", "#DD8452", "#55A868", "#C44E52", "#8172B2",
    "#937860", "#DA8BC3", "#8C8C8C", "#CCB974", "#64B5CD",
    "#1F77B4", "#FF7F0E", "#2CA02C", "#D62728",
]

## Step 1 -- who is "the population" here, and which diagnoses do we even consider?

Unlike earlier notebooks, this one is **not** scoped to one prediction task's cohort (`guo_los` or `new_hypertension`). Disease progression is a population-level relationship, not something specific to one prediction question -- so this uses **every patient** in the full local database (6,731 people), and considers only the ~200 most common diagnoses among them. Restricting to the top 200 (out of ~17,800 distinct codes that exist in the data) keeps the eventual graph readable and keeps every relative-risk estimate resting on a reasonable amount of real data, instead of drowning in thousands of one-off rare codes.

In [ ]:
db = open_database(str(meds_reader_path))
all_patient_ids = list(db)
n_population = len(all_patient_ids)

condition_counts = patient_code_counts(db, all_patient_ids, table="condition")
top200_codes = [code for code, _ in sorted(condition_counts.items(), key=lambda kv: -kv[1])[:200]]
print(f"population: {n_population:,} patients")
print(f"distinct condition codes seen at all: {len(condition_counts):,}")
min_support_count = condition_counts[top200_codes[-1]]
print(
    f"keeping the top 200 -- the 200th most common one is present in "
    f"{min_support_count} patients"
)

In [ ]:
concept_table = load_concept_table(assets / "femr" / "logs" / "omop_dir" / "concept.csv")
resolved = resolve_codes(pd.Series(top200_codes), concept_table).set_index("code")
code_names = resolved["concept_name"].fillna(pd.Series(top200_codes, index=top200_codes)).to_dict()
print(f"name-resolution match rate: {resolved['concept_name'].notna().mean():.1%}")

top15 = pd.Series({code_names[c]: condition_counts[c] for c in top200_codes[:15]})
fig, ax = plt.subplots(figsize=(8, 6))
top15.iloc[::-1].plot(kind="barh", ax=ax, color=PALETTE[0])
ax.set_xlabel("number of patients")
ax.set_title(
    "The 15 most common diagnoses in the population\n"
    "(these become the highest-support nodes in the graph)"
)
fig.tight_layout()
plt.show()

## Step 2 -- when did each patient first get each diagnosis?

We need each patient's diagnosis *timeline*, not just which diagnoses they have. Specifically, the **earliest** time each of the 200 diagnoses appears for them -- if a diagnosis is recorded 30 times across 30 visits, that's one onset, not 30. `first_occurrence_times` does exactly this, restricted to the `condition` table.

In [ ]:
first_times = first_occurrence_times(
    db, all_patient_ids, table="condition", codes_of_interest=set(top200_codes)
)
codes_per_patient = [len(v) for v in first_times.values()]
print(f"mean top-200 diagnoses per patient: {np.mean(codes_per_patient):.1f}")
print(f"patients with >=2 of the top-200 diagnoses (needed to form any trajectory): "
      f"{sum(1 for n in codes_per_patient if n >= 2):,} of {n_population:,}")

## Step 3 -- compute relative risk for every diagnosis pair

For every ordered pair (A, B), we look at every patient who has A, and check whether B showed up strictly *after* A, within a **1-year window**. That window size is a real modeling decision (not a default from a library) -- 1 year is long enough to capture genuine downstream progression, short enough that "B happened decades later, probably unrelated" pairs don't get counted as a trajectory.

In [ ]:
from datetime import timedelta

WINDOW = timedelta(days=365)
MIN_SUPPORT = 10

rr_df = compute_relative_risks(
    first_times,
    codes=top200_codes,
    window=WINDOW,
    min_support=MIN_SUPPORT,
    n_population=n_population,
)
print(f"{len(rr_df):,} ordered diagnosis pairs have at least {MIN_SUPPORT} patients going A-then-B")
rr_df[["a", "b", "n_a", "n_a_then_b", "rr", "pvalue"]].sort_values("rr", ascending=False).head(5)

## Step 4 -- only keep the pairs that are statistically real

A raw significance threshold (like the usual 0.05) isn't strict enough here -- we're testing ~40,000 possible diagnosis pairs at once (200 x 199), and at a 5% false-positive rate, that's up to *2,000* pairs that could look "significant" by pure chance alone. The **Bonferroni correction** divides the significance bar by the number of things being tested, making it dramatically stricter -- the price of testing many hypotheses at once instead of just one.

In [ ]:
N_POSSIBLE_PAIRS = 200 * 199
raw_alpha = 0.05
bonferroni_alpha = raw_alpha / N_POSSIBLE_PAIRS
print(f"raw significance threshold: {raw_alpha}")
print(
    f"Bonferroni-corrected threshold: {bonferroni_alpha:.2e}  "
    f"(over {N_POSSIBLE_PAIRS:,} possible pairs)"
)

significant_df = bonferroni_significant(rr_df, n_codes=200, alpha=raw_alpha)
print(f"pairs surviving the corrected threshold: {len(significant_df):,}")

In [ ]:
# The "have >=1 A-then-B patient" stage needs its own pass with
# min_support=1 -- rr_df above already has the min_support=10 filter
# baked in, so it can't tell us how many pairs existed before that cut.
all_pairs_df = compute_relative_risks(
    first_times, codes=top200_codes, window=WINDOW, min_support=1, n_population=n_population
)

funnel_labels = [
    "All possible\nordered pairs\n(200 x 199)",
    "Have >=1\nA-then-B patient",
    f"Have >={MIN_SUPPORT}\nA-then-B patients",
    "Statistically\nsignificant\n(Bonferroni)",
]
funnel_values = [N_POSSIBLE_PAIRS, len(all_pairs_df), len(rr_df), len(significant_df)]

fig, ax = plt.subplots(figsize=(9, 4.5))
bar_colors = [PALETTE[7], PALETTE[9], PALETTE[0], PALETTE[3]]
bars = ax.bar(funnel_labels, funnel_values, color=bar_colors)
for bar, value in zip(bars, funnel_values, strict=True):
    ax.text(
        bar.get_x() + bar.get_width() / 2, bar.get_height(), f"{value:,}",
        ha="center", va="bottom", fontsize=10,
    )
ax.set_yscale("log")
ax.set_ylabel("number of ordered pairs (log scale)")
ax.set_title("How many candidate diagnosis pairs survive each filtering stage")
fig.tight_layout()
plt.show()

This is worth sitting with: out of **39,800** possible directed relationships between the top 200 diagnoses, only a few hundred hold up as real, well-supported, statistically significant trajectories. That's the whole point of the filtering -- a graph with 39,800 edges would be meaningless noise; this one is small enough that every edge represents something real.

## Step 5 -- build the graph, with real diagnosis names

In [ ]:
G = build_trajectory_graph(significant_df, code_names=code_names)
print(f"directed graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")
print(
    f"({200 - G.number_of_nodes()} of the top-200 diagnoses have no significant trajectory "
    "in or out of them at all, given our thresholds -- they simply didn't participate in any "
    "strong-enough pattern, which is itself worth knowing, not a gap to paper over)"
)

G_undirected = symmetrize_max(G)
print(f"undirected version (for community coloring below): {G_undirected.number_of_edges()} edges")

## Step 6 -- the graph itself

This is the centerpiece. Nodes are diagnoses (real names, not codes); an edge means one diagnosis significantly raises the risk of another within a year. Colors are **communities** -- groups of diagnoses that cluster tightly together, found automatically (Louvain method) purely to make the picture more readable and to preview what issue #14's more rigorous clustering (Girvan-Newman) will explore in depth. Node size reflects how many other diagnoses connect to it (its degree) -- bigger nodes are more "central" to disease progression in this population.

**One layout decision worth explaining:** this graph isn't fully connected -- it naturally splits into separate "islands" of diagnoses that never significantly predict anything in another island. Cramming every island into one shared layout (the default behavior of a force-directed layout on a disconnected graph) wastes most of the canvas on empty space between islands and gives no real information, since there's nothing to "attract" them together. Instead, the two large islands get their own dedicated, properly laid-out panel each; the handful of tiny 2-3-node islands are listed directly below instead of plotted, since a "network diagram" of one pair of nodes is just a labeled line -- more legible as text.

In [ ]:
partition = community_louvain.best_partition(G_undirected, weight="weight", random_state=0)
n_communities = len(set(partition.values()))
components = sorted(nx.connected_components(G_undirected), key=len, reverse=True)
big_components = [c for c in components if len(c) >= 10]
small_components = [c for c in components if len(c) < 10]
print(f"{n_communities} communities found (via Louvain, for coloring)")
print(f"{len(components)} separate connected islands total")
print(f"{len(big_components)} large islands (sizes: {[len(c) for c in big_components]})")
n_small_nodes = sum(len(c) for c in small_components)
print(
    f"{len(small_components)} small islands, {n_small_nodes} nodes total -- "
    "listed below, not plotted"
)

In [ ]:
degrees = dict(G_undirected.degree())

fig, axes = plt.subplots(1, len(big_components), figsize=(9 * len(big_components), 10))
if len(big_components) == 1:
    axes = [axes]

for ax, comp_nodes in zip(axes, big_components, strict=True):
    subgraph = G_undirected.subgraph(comp_nodes)
    layout = nx.kamada_kawai_layout(subgraph, weight="weight")

    node_colors = [PALETTE[partition[n] % len(PALETTE)] for n in subgraph.nodes]
    node_sizes = [120 + degrees[n] * 55 for n in subgraph.nodes]

    nx.draw_networkx_edges(subgraph, layout, alpha=0.3, width=0.9, edge_color="#999999", ax=ax)
    nx.draw_networkx_nodes(
        subgraph, layout, node_size=node_sizes, node_color=node_colors,
        alpha=0.92, linewidths=0.6, edgecolors="white", ax=ax,
    )

    top_labeled = sorted(subgraph.nodes, key=lambda n: -degrees[n])[:14]
    nx.draw_networkx_labels(
        subgraph, layout,
        labels={n: subgraph.nodes[n]["name"][:26] for n in top_labeled},
        font_size=8.5, ax=ax,
        bbox=dict(facecolor="white", alpha=0.8, edgecolor="none", pad=0.4),
    )
    ax.set_title(f"Island of {len(comp_nodes)} diagnoses", fontsize=12)
    ax.axis("off")

fig.suptitle(
    "Disease-trajectory graph -- the two largest connected islands "
    f"({sum(len(c) for c in big_components)} of {G_undirected.number_of_nodes()} diagnoses)\n"
    "node size = number of connections, color = automatically-detected community, "
    "labels = 14 most-connected per island",
    fontsize=12,
)
fig.tight_layout()
plt.show()

print("\nThe remaining small islands (too small to usefully plot as a network):")
for comp_nodes in small_components:
    names = [G_undirected.nodes[n]["name"] for n in comp_nodes]
    print(f"  - {', '.join(names)}")

## Step 7 -- a readable, *directed* zoom-in

The picture above deliberately drops arrow directions (Girvan-Newman, its eventual destination, needs an undirected graph). But direction is the entire point of a trajectory graph -- so here's a genuinely directed view, arrows and all, so the "A leads to B" story is actually visible somewhere in this notebook. Two readability decisions, both the same lesson from Step 6 applied again: stay within one connected island (not edges scattered across disconnected fragments), and keep only the *strongest* links (the full island has 309 directed edges -- too dense to read arrow-by-arrow) rather than plotting all of them.

In [ ]:
largest_island = big_components[0]
island_edges = [
    (u, v, d) for u, v, d in G.edges(data=True) if u in largest_island and v in largest_island
]
top_island_edges = sorted(island_edges, key=lambda e: -e[2]["weight"])[:40]

G_top = nx.DiGraph()
for u, v, d in top_island_edges:
    for c in (u, v):
        if c not in G_top:
            G_top.add_node(c, name=code_names[c])
    G_top.add_edge(u, v, weight=d["weight"])
top_degrees = dict(G_top.degree())

top_layout = nx.kamada_kawai_layout(G_top, weight="weight")

fig, ax = plt.subplots(figsize=(13, 12))
node_sizes = [200 + top_degrees[n] * 90 for n in G_top.nodes]
nx.draw_networkx_nodes(
    G_top, top_layout, node_size=node_sizes, node_color=PALETTE[0],
    alpha=0.9, edgecolors="white", linewidths=0.6, ax=ax,
)
nx.draw_networkx_edges(
    G_top, top_layout, alpha=0.55, width=1.2, edge_color="#777777",
    arrows=True, arrowsize=12, arrowstyle="-|>", connectionstyle="arc3,rad=0.08", ax=ax,
)
top_labeled = sorted(G_top.nodes, key=lambda n: -top_degrees[n])[:22]
nx.draw_networkx_labels(
    G_top, top_layout,
    labels={n: G_top.nodes[n]["name"][:26] for n in top_labeled},
    font_size=8, ax=ax,
    bbox=dict(facecolor="white", alpha=0.82, edgecolor="none", pad=0.4),
)
ax.set_title(
    f"Directed view of the largest island ({G_top.number_of_nodes()} diagnoses, "
    f"{G_top.number_of_edges()} directed links) -- arrows show A -> B",
    fontsize=12,
)
ax.axis("off")
fig.tight_layout()
plt.show()

## Step 8 -- which diagnoses are "triggers," and which are "destinations"?

**Out-degree** (many arrows leaving) = a diagnosis that tends to *lead to* many other things -- an early warning sign. **In-degree** (many arrows arriving) = a diagnosis that many different conditions tend to *end up at* -- a common downstream consequence, almost a "final common pathway."

In [ ]:
out_degree = pd.Series(dict(G.out_degree())).sort_values(ascending=False).head(12)
in_degree = pd.Series(dict(G.in_degree())).sort_values(ascending=False).head(12)

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
pd.Series({code_names[c]: v for c, v in out_degree.items()}).iloc[::-1].plot(
    kind="barh", ax=axes[0], color=PALETTE[2]
)
axes[0].set_title("Top \"triggers\"\n(most other diagnoses they lead to)")
axes[0].set_xlabel("out-degree")

pd.Series({code_names[c]: v for c, v in in_degree.items()}).iloc[::-1].plot(
    kind="barh", ax=axes[1], color=PALETTE[3]
)
axes[1].set_title("Top \"destinations\"\n(most other diagnoses that lead to them)")
axes[1].set_xlabel("in-degree")

fig.tight_layout()
plt.show()

These make real clinical sense as a sanity check: the top "destinations" cluster around pulmonary/cardiac decompensation (pulmonary congestion, pulmonary edema, pleural effusion) -- plausible late-stage, downstream complications that many different underlying conditions can lead to, not a random assortment of unrelated codes. That's a good sign the graph is capturing something real, not noise.

## Summary -- issue #10 done-when checklist

- **The disease-trajectory graph exists, with a documented construction rule** (this was previously entirely unspecified -- see the module docstring in `ehrgraph/graphs/trajectory.py` and Step 1-5 above): nodes = top 200 condition codes by patient count across the full population; edges = statistically significant (Bonferroni-corrected), well-supported (>=10 patients), 1-year-window relative risk.
- **Symmetrized via `max(RR_AB, RR_BA)`**, ready for Girvan-Newman (issue #14) -- with the directionality tradeoff stated explicitly, not silently dropped (Step 7 shows the real directed structure separately, so it isn't lost from the notebook entirely).
- Every stage (`first_occurrence_times`, `compute_relative_risks`, `bonferroni_significant`, `build_trajectory_graph`, `symmetrize_max`) is independently unit-tested in `tests/test_trajectory.py`, and re-verified here against real data.
- Real, honest reporting: 65 of the top-200 diagnoses have no significant trajectory at all under these thresholds; the graph's most-connected "destination" nodes are plausible late-stage cardiopulmonary complications, which is a real, checkable sanity signal that the graph reflects genuine clinical patterns rather than noise.